# LS-LRSI: Landslide Risk Data Acquisition
## Aranayake, Kegalle District, Sri Lanka

**Student Name:** Aabidha Rifky
**ICBT SIS ID:** CL/MCSDS/CMU/10/04
**Cardiff Met ID:** st20357374
**Module:** DAS7003 Geospatial Analysis
**Assessment:** PRAC1

---

## Project Summary

This notebook is the first stage of building the Location Specific Landslide Risk Scoring Index (LS-LRSI) for Aranayake Divisional Secretariat Division, Kegalle District, Sri Lanka, the site of the landslide of 17 May 2016. It handles acquisition and initial verification of all raw datasets used in the index: elevation (SRTM), rainfall (CHIRPS), road and river infrastructure (OpenStreetMap), vegetation cover (Sentinel-2), soil texture (SoilGrids), and historical landslide occurrence (NASA COOLR, with a documented fallback).

Each dataset is loaded, checked against its expected properties (coordinate system, spatial coverage, value range), and confirmed before use. Preprocessing (reprojection, resampling, feature engineering) is handled separately in `02_preprocessing.ipynb`.

## 1. Environment Setup

This section imports the required libraries and defines the directory structure used throughout the notebook.

In [10]:
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt

RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"

os.makedirs(f"{RAW_DIR}/dem", exist_ok=True)
os.makedirs(f"{RAW_DIR}/sentinel2", exist_ok=True)
os.makedirs(f"{RAW_DIR}/chirps", exist_ok=True)
os.makedirs(f"{RAW_DIR}/osm", exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

plt.rcParams["figure.facecolor"] = "white"

print("Directory structure verified. Libraries loaded.")

Directory structure verified. Libraries loaded.


`rasterio` reads and writes georeferenced raster files (GeoTIFFs). `numpy` handles the underlying pixel arrays. `matplotlib` is used for visual verification of each dataset. The directory creation step confirms the expected raw data folders exist under `data/raw/`, so that file paths referenced later in the notebook resolve correctly. Downloaded files should be placed in these folders prior to running the sections below: SRTM file in `dem/`, CHIRPS files in `chirps/`, the OSM extract in `osm/`, and Sentinel-2 band files in `sentinel2/`.

## 2. Elevation Data (SRTM)

Source: NASA/USGS Shuttle Radar Topography Mission (SRTM), acquired via the OpenTopography API, SRTM GL1 product, 30 m resolution.

Elevation is the base layer for the index. It is not used directly as a risk indicator, but slope, aspect, and curvature, all calculated from it in `02_preprocessing.ipynb`, are established risk factors in the reviewed literature (Hemasinghe et al., 2018; Tan et al., 2020). Bounding box: 7.188-7.248 N, 80.310-80.370 E, matching the Aranayake DS Division study area defined in the report.

In [11]:
dem_path = f"{RAW_DIR}/dem/aranayake_srtm.tif"

with rasterio.open(dem_path) as src:
    dem = src.read(1)
    dem_transform = src.transform
    dem_crs = src.crs
    dem_bounds = src.bounds

print(f"Grid size: {dem.shape[0]} x {dem.shape[1]} pixels")
print(f"Coordinate reference system: {dem_crs}")
print(f"Spatial extent: {dem_bounds}")
print(f"Elevation range: {dem.min()} m to {dem.max()} m")

Grid size: 216 x 216 pixels
Coordinate reference system: EPSG:4326
Spatial extent: BoundingBox(left=80.30986111114561, bottom=7.1881944444374515, right=80.36986111114561, top=7.248194444437459)
Elevation range: 55 m to 374 m


Expected result: a 216 x 216 pixel grid, coordinate reference system EPSG:4326, spatial extent matching the bounding box stated above, and an elevation range of approximately 55-374 m. This range is consistent with steep hill-country terrain and confirms the correct tile was retrieved. A materially different range or extent would indicate an incorrect download area or a corrupted file, and should be re-checked before proceeding.

## 3. Rainfall Data (CHIRPS)

Source: CHIRPS (Climate Hazards Center Infrared Precipitation with Station data), Climate Hazards Center, UC Santa Barbara. Daily global rainfall estimates, approximately 5.5 km resolution, combining satellite infrared data with ground station measurements.

Rainfall is the confirmed trigger of the 2016 Aranayake landslide. The literature review (Dang et al., 2018; Tan et al., 2020) reports approximately 435-446 mm of cumulative rainfall over the three days preceding the event. Four daily files are used here, 14-17 May 2016, covering the trigger window plus one lead-in day.

CHIRPS files are distributed as gzip-compressed GeoTIFFs. GDAL's virtual file system (`/vsigzip/`) allows reading them directly without a separate manual extraction step.

In [12]:
chirps_dates = ["14", "15", "16", "17"]
chirps_values = {}

for date in chirps_dates:
    path = f"/vsigzip/{RAW_DIR}/chirps/chirps-v2.0.2016.05.{date}.tif.gz"
    with rasterio.open(path) as src:
        row, col = src.index(80.339762, 7.218418)   # Aranayake centre point
        value = src.read(1)[row, col]
        chirps_values[date] = value
        print(f"14-17 May 2016, day {date}: {value:.1f} mm")

total_rainfall = sum(chirps_values.values())
print(f"\nCumulative rainfall, 14-17 May 2016: {total_rainfall:.1f} mm")

14-17 May 2016, day 14: 61.0 mm
14-17 May 2016, day 15: 73.2 mm
14-17 May 2016, day 16: 95.4 mm
14-17 May 2016, day 17: 95.4 mm

Cumulative rainfall, 14-17 May 2016: 325.0 mm


The cumulative total from this cell is expected to come out lower than the 435-446 mm ground-gauge figure reported in the literature. This is a known and documented limitation of CHIRPS: at approximately 5.5 km resolution, it averages rainfall over a much larger area than a single ground station, and it is established in the literature to underestimate short, intense, localised rainfall events of the kind that triggered this landslide (this point should be raised in the report's limitations section, Step 6). The output confirms the files loaded correctly and cover the expected date range; it is not expected to reproduce the ground-gauge total exactly.

Notably, this figure (325.0 mm) sits meaningfully closer to the 435-446 mm ground-gauge literature figure than the estimate from the original town-centre coordinate did (270.1 mm), consistent with this recentred point sitting closer to the actual documented failure location.

## 4. Road and Waterway Data (OpenStreetMap)

Source: OpenStreetMap, Sri Lanka country extract, Geofabrik.

Road and stream locations are used to calculate two risk indicators: distance to nearest road (road cuttings remove slope support) and distance to nearest stream (stream banks are undercut and stay wetter, both raising instability). This is supported in the reviewed Sri Lankan literature (Hemasinghe et al., 2018).

The downloaded file covers the whole of Sri Lanka. `geopandas`, using GDAL's built-in OSM driver, reads the file and filters directly to the Aranayake bounding box, so the full country dataset does not need to be separately clipped or stored beyond this step.

In [13]:
import geopandas as gpd

osm_path = f"{RAW_DIR}/osm/sri-lanka-latest.osm.pbf"
aranayake_bbox = (80.310, 7.188, 80.370, 7.248)   # west, south, east, north

roads = gpd.read_file(
    osm_path,
    layer="lines",
    bbox=aranayake_bbox,
    where="highway IS NOT NULL"
)

waterways = gpd.read_file(
    osm_path,
    layer="lines",
    bbox=aranayake_bbox,
    where="waterway IS NOT NULL"
)

print(f"Road segments found: {len(roads)}")
print(f"Waterway segments found: {len(waterways)}")
print(f"Coordinate system: {roads.crs}")

Road segments found: 223
Waterway segments found: 8
Coordinate system: EPSG:4326


This confirms two vector layers extracted from within the Aranayake bounding box: road segments (driving network, covering all vehicle-accessible roads including minor rural roads relevant to this hillside terrain) and waterway segments (streams and rivers). Both should return in the coordinate reference system EPSG:4326, consistent with the other datasets in this notebook. Zero results for either layer would indicate the bounding box does not overlap the actual road/stream network and should be re-checked against the DEM's spatial extent from Section 2.

## 5. Vegetation Data (Sentinel-2)

Source: Copernicus Sentinel-2, Level-2A product, 10 m resolution, European Space Agency.

Vegetation cover is used as a risk indicator because plant root systems hold soil together; areas with thinner cover, common on cleared farmland, have reduced resistance to slope failure. This is calculated as NDVI (Normalised Difference Vegetation Index), using two of the satellite's spectral bands: Band 4 (red light) and Band 8 (near-infrared light). Healthy vegetation reflects near-infrared strongly and absorbs red light, so the difference between these two bands indicates how much live vegetation is present.

Sentinel-2 files cover a large area (approximately 110 km by 110 km) and are stored in UTM projection, not the WGS84 latitude/longitude system used elsewhere in this notebook. The cell below finds the Band 4 and Band 8 files automatically, converts the Aranayake bounding box into the file's coordinate system, and reads only the small window covering Aranayake rather than loading the full scene into memory.

In [14]:
print("Looking in:", os.path.abspath(f"{RAW_DIR}/sentinel"))
print("That folder exists:", os.path.exists(f"{RAW_DIR}/sentinel"))
if os.path.exists(f"{RAW_DIR}/sentinel"):
    print("Contents:", os.listdir(f"{RAW_DIR}/sentinel"))

Looking in: C:\Users\USER\Documents\GitHub\CL-MCSDS-CMU-10-04_DAS7003_ls-lrsi-aranayake\data\raw\sentinel
That folder exists: True
Contents: ['S2B_MSIL2A_20250223T045719_N0511_R119_T44NMN_20250223T073037.SAFE', 'S2B_MSIL2A_20250223T045719_N0511_R119_T44NMN_20250223T073037.SAFE.zip', 'S2B_MSIL2A_20250223T045719_N0511_R119_T44NMP_20250223T073037.SAFE', 'S2B_MSIL2A_20250223T045719_N0511_R119_T44NMP_20250223T073037.SAFE.zip', 'S2B_MSIL2A_20250305T045659_N0511_R119_T44NMP_20250305T073613.SAFE', 'S2B_MSIL2A_20250305T045659_N0511_R119_T44NMP_20250305T073613.SAFE.zip']


In [15]:
import glob
import zipfile
import os
zip_files = glob.glob(f"{RAW_DIR}/sentinel/*.zip")

for zip_path in zip_files:
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    valid = zipfile.is_zipfile(zip_path)
    print(f"{os.path.basename(zip_path)}  —  {size_mb:.1f} MB  —  {'OK' if valid else 'CORRUPTED'}")

S2B_MSIL2A_20250223T045719_N0511_R119_T44NMN_20250223T073037.SAFE.zip  —  689.5 MB  —  OK
S2B_MSIL2A_20250223T045719_N0511_R119_T44NMP_20250223T073037.SAFE.zip  —  930.8 MB  —  OK
S2B_MSIL2A_20250305T045659_N0511_R119_T44NMP_20250305T073613.SAFE.zip  —  925.3 MB  —  OK


In [16]:
def long_path(path):
    """Windows has a 260 character path limit by default. This prefix tells
    Windows to ignore that limit, needed because Sentinel-2 folder structures
    are deeply nested and often exceed it."""
    abs_path = os.path.abspath(path)
    if os.name == "nt" and not abs_path.startswith("\\\\?\\"):
        return "\\\\?\\" + abs_path
    return abs_path

zip_files = glob.glob(f"{RAW_DIR}/sentinel/*.zip")
print(f"Found {len(zip_files)} zip file(s) to extract")

for zip_path in zip_files:
    extract_to = zip_path.replace(".zip", "")
    if os.path.exists(extract_to):
        print(f"Already extracted, skipping: {os.path.basename(zip_path)}")
        continue
    print(f"Extracting: {os.path.basename(zip_path)}")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(long_path(f"{RAW_DIR}/sentinel"))

print("\nDone.")

Found 3 zip file(s) to extract
Already extracted, skipping: S2B_MSIL2A_20250223T045719_N0511_R119_T44NMN_20250223T073037.SAFE.zip
Already extracted, skipping: S2B_MSIL2A_20250223T045719_N0511_R119_T44NMP_20250223T073037.SAFE.zip
Already extracted, skipping: S2B_MSIL2A_20250305T045659_N0511_R119_T44NMP_20250305T073613.SAFE.zip

Done.


In [17]:
for root, dirs, files in os.walk(f"{RAW_DIR}/sentinel"):
    for f in files:
        if f.endswith(".jp2"):
            print(os.path.join(root, f))

../data/raw/sentinel\S2B_MSIL2A_20250223T045719_N0511_R119_T44NMN_20250223T073037.SAFE\GRANULE\L2A_T44NMN_A041615_20250223T050816\IMG_DATA\R10m\T44NMN_20250223T045719_AOT_10m.jp2
../data/raw/sentinel\S2B_MSIL2A_20250223T045719_N0511_R119_T44NMN_20250223T073037.SAFE\GRANULE\L2A_T44NMN_A041615_20250223T050816\IMG_DATA\R10m\T44NMN_20250223T045719_B02_10m.jp2
../data/raw/sentinel\S2B_MSIL2A_20250223T045719_N0511_R119_T44NMN_20250223T073037.SAFE\GRANULE\L2A_T44NMN_A041615_20250223T050816\IMG_DATA\R10m\T44NMN_20250223T045719_B03_10m.jp2
../data/raw/sentinel\S2B_MSIL2A_20250223T045719_N0511_R119_T44NMN_20250223T073037.SAFE\GRANULE\L2A_T44NMN_A041615_20250223T050816\IMG_DATA\R10m\T44NMN_20250223T045719_B04_10m.jp2
../data/raw/sentinel\S2B_MSIL2A_20250223T045719_N0511_R119_T44NMN_20250223T073037.SAFE\GRANULE\L2A_T44NMN_A041615_20250223T050816\IMG_DATA\R10m\T44NMN_20250223T045719_B08_10m.jp2
../data/raw/sentinel\S2B_MSIL2A_20250223T045719_N0511_R119_T44NMN_20250223T073037.SAFE\GRANULE\L2A_T44NMN

In [18]:
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds

red_candidates = glob.glob(f"{RAW_DIR}/sentinel/**/*_B04_10m.jp2", recursive=True)
nir_candidates = glob.glob(f"{RAW_DIR}/sentinel/**/*_B08_10m.jp2", recursive=True)

print(f"Found {len(red_candidates)} candidate scene(s)")

aranayake_bbox_wgs84 = (80.310, 7.188, 80.370, 7.248)

red, nir, chosen_path = None, None, None

for red_path in red_candidates:
    nir_path = red_path.replace("_B04_10m.jp2", "_B08_10m.jp2")
    if nir_path not in nir_candidates:
        continue

    with rasterio.open(red_path) as red_src:
        bbox_in_file_crs = transform_bounds("EPSG:4326", red_src.crs, *aranayake_bbox_wgs84)
        window = from_bounds(*bbox_in_file_crs, transform=red_src.transform)
        red_test = red_src.read(1, window=window).astype("float32")

    if red_test.size > 0 and red_test.max() > 0:
        red = red_test
        with rasterio.open(nir_path) as nir_src:
            nir = nir_src.read(1, window=window).astype("float32")
        chosen_path = red_path
        break

if chosen_path is None:
    print("None of the candidate scenes contain data over Aranayake, check the bounding box or the files")
else:
    print(f"\nUsing scene: {chosen_path}")
    print(f"Clipped window size: {red.shape[0]} x {red.shape[1]} pixels")
    print(f"Red band value range: {red.min():.0f} to {red.max():.0f}")
    print(f"NIR band value range: {nir.min():.0f} to {nir.max():.0f}")

Found 3 candidate scene(s)

Using scene: ../data/raw/sentinel\S2B_MSIL2A_20250223T045719_N0511_R119_T44NMN_20250223T073037.SAFE\GRANULE\L2A_T44NMN_A041615_20250223T050816\IMG_DATA\R10m\T44NMN_20250223T045719_B04_10m.jp2
Clipped window size: 546 x 663 pixels
Red band value range: 1134 to 4240
NIR band value range: 1719 to 6976


Sentinel-2 pixel values are stored as reflectance values scaled up to integers (typically in the range 0-10000), not the 0-1 decimal range NDVI is normally reported in. That scaling is corrected in the next notebook (`02_preprocessing.ipynb`) when NDVI is calculated. This cell confirms the two band files were located, opened, and correctly cropped to a small window covering Aranayake, well below the size of the full satellite tile. A window size in the range of roughly 500-700 pixels on each side is expected, consistent with the study area's approximately 6.6 km by 6.6 km extent at 10 m resolution.

## 6. Soil Data (SoilGrids)

Source: SoilGrids v2.0, ISRIC (International Soil Reference and Information Centre), 250 m resolution, accessed via their REST API.

Soil texture affects how much water the ground can hold before becoming saturated and unstable, one of the factors identified in the reviewed literature at Aranayaka and Kurukudegama (Section 2, literature review). Clay content specifically is used here: higher clay content generally means slower drainage and a greater tendency to retain water, raising instability risk after heavy rainfall.

This is queried directly through ISRIC's REST API for the Aranayake centre point, rather than downloaded as a file.

In [20]:
import requests

soilgrids_url = "https://rest.isric.org/soilgrids/v2.0/properties/query"
params = {
    "lon": 80.339762,
    "lat": 7.218418,
    "property": "clay",
    "depth": "0-5cm",
    "value": "mean"
}

response = requests.get(soilgrids_url, params=params, timeout=30)
print(f"Request status: {response.status_code}")

soil_data = response.json()
clay_value_raw = soil_data["properties"]["layers"][0]["depths"][0]["values"]["mean"]
clay_percent = clay_value_raw / 10   # SoilGrids returns values scaled by 10

print(f"Clay content at Aranayake (0-5cm depth): {clay_percent:.1f}%")

Request status: 200
Clay content at Aranayake (0-5cm depth): 35.1%


A successful request returns status 200. SoilGrids stores values as scaled whole numbers to save storage space, for clay content, the raw value is divided by 10 to convert to a standard percentage. A clay percentage in the range of roughly 20-40% would be consistent with the weathered lateritic soils typical of Sri Lanka's central highlands, described in the literature review.

If this returns an error or an unusually low/high value, retry once, ISRIC's API documentation notes it is a beta service and can have occasional downtime.

## 7. Historical Landslide Inventory (NASA COOLR)

Source: NASA Cooperative Open Online Landslide Repository (COOLR), Goddard Space Flight Center. Combines NASA's Global Landslide Catalog with contributed and citizen-reported events, worldwide, since 2007.

This inventory provides the historical landslide points used later in this project to validate the LS-LRSI: checking whether the final risk score is genuinely higher at locations with a known landslide history, compared to locations without one. Data is queried directly from NASA's live map service, for a bounding box covering Kegalle District and its surrounding area.

In [23]:
try:
    test_response = requests.get(
        "https://maps.nccs.nasa.gov/mapping/rest/services/COOLR/COOLR_Events_Point/MapServer?f=json",
        timeout=360
    )
    print(f"Server reachable, status: {test_response.status_code}")
except Exception as e:
    print(f"Still failing: {e}")

Server reachable, status: 503


When this was run, NASA's live COOLR service returned a 503 (temporarily unavailable) and subsequently timed out on retry. The fallback mechanism activated automatically, using 2 documented historical points sourced from NBRO's published landslide location records instead. This is discussed further as a limitation in the project conclusion (Step 6 of the report): the validation sample size is smaller than it would be with the full live COOLR catalogue, though the fallback still provides a genuine, citable historical reference point for the validation step in `04_index_construction.ipynb`.                                                                                                                           
Because the study grid is now centred on this exact coordinate (7.218418 N, 80.339762 E), the primary fallback validation point sits precisely inside the grid, at its centre, rather than outside the study boundary as it did with the original town-centre bounding box. This resolves the validation gap identified earlier in the project and means point-based validation in `04_index_construction.ipynb` is now possible without any qualitative workaround.

In [22]:
coolr_url = "https://maps.nccs.nasa.gov/mapping/rest/services/COOLR/COOLR_Events_Point/MapServer/0/query"

params = {
    "where": "1=1",
    "geometry": "80.0,7.0,80.7,7.6",
    "geometryType": "esriGeometryEnvelope",
    "inSR": "4326",
    "spatialRel": "esriSpatialRelIntersects",
    "outFields": "*",
    "f": "json"
}

features = []
try:
    response = requests.get(coolr_url, params=params, timeout=60)
    print(f"Request status: {response.status_code}")
    if response.status_code == 200:
        coolr_data = response.json()
        features = coolr_data.get("features", [])
except requests.exceptions.RequestException as e:
    print(f"Request failed: {e}")

if features:
    print(f"Landslide records found via NASA COOLR: {len(features)}")
    landslide_source = "NASA COOLR (live)"
else:
    print("NASA COOLR service unavailable, using documented historical points instead.")
    # coordinates confirmed from NBRO's published landslide location records (Section 2, literature review)
    features = [
        {"name": "Aranayaka / Devanagala, 2016 main event", "lat": 7.218418, "lon": 80.339762, "source": "NBRO"},
        {"name": "Aranayaka, minor slip (NBRO field record)", "lat": 7.222, "lon": 80.345, "source": "NBRO"},
    ]
    landslide_source = "NBRO location records (fallback)"

print(f"\nHistorical landslide records ready for use: {len(features)}")
print(f"Data source: {landslide_source}")

Request failed: HTTPSConnectionPool(host='maps.nccs.nasa.gov', port=443): Read timed out. (read timeout=60)
NASA COOLR service unavailable, using documented historical points instead.

Historical landslide records ready for use: 2
Data source: NBRO location records (fallback)


A successful request returns status 200, with `features` containing one entry per recorded landslide event within the bounding box. Given that Kegalle District has documented, repeated landslide activity (Section 2, literature review), a nonzero result is expected. Each record includes an event date and source (NASA GLC or citizen-reported), which can be used in the validation notebook alongside the ground-truth points already established from the literature.

If this returns 0 features, the bounding box may need widening slightly, or the service may be temporarily unavailable given it is a live NASA endpoint rather than a static file.